# PPY - Class 07: `singledispatch` & Magic Methods
**Topics:** `@singledispatch`, `@singledispatchmethod`, dunder methods, operator overloading

## 🧠 Theory

### singledispatch
```python
from functools import singledispatch

@singledispatch
def describe(value):
    return f"Default: {value}"

@describe.register(int)
def _(value): return f"Integer: {value}"

@describe.register(str)
def _(value): return f"String: {value}"

describe(10)      # -> "Integer: 10"
describe("hi")    # -> "String: hi"
describe(3.14)    # -> "Default: 3.14"
```

### singledispatchmethod (in classes)
```python
from functools import singledispatchmethod

class Processor:
    @singledispatchmethod
    def process(self, value):
        return "Unknown"

    @process.register(int)
    def _(self, value): return value * 2

    @process.register(str)
    def _(self, value): return value.upper()
```

### Key dunder methods
```python
__str__ / __repr__          # string representation
__len__                     # len(obj)
__bool__                    # bool(obj)
__call__                    # obj()
__add__ / __sub__ / __mul__ # arithmetic
__matmul__                  # @ operator
__lt__ / __eq__             # comparison
__iter__ / __next__         # iteration
__contains__                # in operator
__getitem__ / __setitem__   # indexing
__enter__ / __exit__        # context manager
```

## ✏️ singledispatch I-VIII – function dispatch

In [ ]:
from functools import singledispatch

# process()
@singledispatch
def process(value): return "Unsupported"

@process.register(int)
def _(v): return v * 2

@process.register(str)
def _(v): return v.upper()

print("process(5):      ", process(5))
print("process('hello'):", process("hello"))
print("process(3.14):   ", process(3.14))

# describe()
@singledispatch
def describe(v): return "Unknown type"

@describe.register(int)
def _(v): return "Integer"

@describe.register(str)
def _(v): return "String"

print("describe(42):  ", describe(42))
print("describe('hi'):", describe("hi"))
print("describe(3.14):", describe(3.14))

# other utilities
@singledispatch
def reverse(v): raise NotImplementedError

@reverse.register(list)
def _(v): return v[::-1]

@reverse.register(str)
def _(v): return v[::-1]

print("reverse('hello'):", reverse("hello"))
print("reverse([1,2,3]):", reverse([1,2,3]))

## ✏️ singledispatchmethod I – Alice: Madness Returns

In [ ]:
from functools import singledispatchmethod

class Cake: pass
class Potion: pass
class HobbyHorse: pass
class PepperGrinder: pass
class VorpalBlade: pass

class Alice:
    def __init__(self, name, size=10):
        self.name   = name
        self.size   = size
        self.weapon = None

    @singledispatchmethod
    def interact(self, target):
        return f"{self.name}: no interaction with {type(target).__name__}"

    @interact.register(Cake)
    def _(self, target):
        self.size += 2
        return f"{self.name} eats Cake -> size now {self.size}"

    @interact.register(Potion)
    def _(self, target):
        self.size -= 2
        return f"{self.name} drinks Potion -> size now {self.size}"

    @interact.register(HobbyHorse)
    @interact.register(PepperGrinder)
    @interact.register(VorpalBlade)
    def _(self, target):
        self.weapon = type(target).__name__
        return f"{self.name} equips {self.weapon}"

    def __str__(self):  return f"Alice | size={self.size} | weapon={self.weapon}"
    def __len__(self):  return self.size
    def __bool__(self): return self.size > 0

alice = Alice("Alice", size=10)
print(alice.interact(Cake()))
print(alice.interact(Potion()))
print(alice.interact(HobbyHorse()))
print(alice.interact(VorpalBlade()))
print(alice)
print("len:", len(alice), "| bool:", bool(alice))

## ✏️ singledispatchmethod II – Minecraft Interaction System

In [ ]:
from functools import singledispatchmethod

class Block:
    def __init__(self, name): self.name = name
class Item:
    def __init__(self, name): self.name = name
    def __repr__(self): return f"Item({self.name})"
class Mob:
    def __init__(self, name, hp): self.name = name; self.hp = hp

class Player:
    def __init__(self, name):
        self.name      = name
        self.inventory = []

    @singledispatchmethod
    def interact(self, target):
        return f"No interaction with {type(target).__name__}"

    @interact.register(Block)
    def _(self, target):
        return f"{self.name} interacts with block: {target.name}"

    @interact.register(Item)
    def _(self, target):
        self.inventory.append(target)
        return f"{self.name} picked up {target.name}"

    @interact.register(Mob)
    def _(self, target):
        target.hp -= 10
        if target.hp <= 0: return f"{self.name} killed {target.name}"
        return f"{self.name} hit {target.name}, HP left: {target.hp}"

    def __str__(self):  return f"Player({self.name}, inventory={self.inventory})"
    def __len__(self):  return len(self.inventory)

player = Player("Steve")
stone  = Block("Stone"); sword = Item("Sword"); zombie = Mob("Zombie", hp=15)
print(player.interact(stone))
print(player.interact(sword))
print(player.interact(zombie))
print(player.interact(zombie))
print(player); print("Items:", len(player))

## ✏️ singledispatchmethod III – SmartAdder

In [ ]:
from functools import singledispatchmethod

class SmartAdder:
    def __init__(self):
        self.counter = 0
        self.lst     = []
        self.text    = ""

    @singledispatchmethod
    def add(self, value): raise TypeError(f"Unsupported: {type(value)}")

    @add.register(int)
    def _(self, value): self.counter += value

    @add.register(list)
    def _(self, value): self.lst.extend(value)

    @add.register(str)
    def _(self, value): self.text += value

    def __str__(self):
        return f"SmartAdder(counter={self.counter}, list={self.lst}, text={self.text!r})"
    def __len__(self):
        return self.counter + len(self.lst) + len(self.text)

sa = SmartAdder()
sa.add(5); sa.add(3)
sa.add([10, 20, 30])
sa.add("hello"); sa.add(" world")
print(sa); print("Total elements:", len(sa))

## ✏️ Magic methods – NieR: Automata Combat System

In [ ]:
class YoRHaUnit:
    def __init__(self, name, energy, damage):
        self.name = name; self.energy = energy; self.damage = damage

    def __call__(self):
        if not self: return f"{self.name} is offline!"
        self.energy -= 10
        return f"{self.name} ATTACK for {self.damage} dmg | energy={self.energy}"

    def __add__(self, other):  return CompositeSystem([self, other])
    def __mul__(self, factor): return BoostedUnit(self, factor)
    def __len__(self):         return self.energy
    def __bool__(self):        return self.energy > 0
    def __str__(self):
        return f"{self.name} | HP: {self.energy} | DMG: {self.damage} | {'ACTIVE' if self else 'OFFLINE'}"

class CompositeSystem:
    def __init__(self, units): self.units = units
    def __call__(self):        return "\n".join(u() for u in self.units)
    def __mul__(self, factor): return BoostedSystem(self.units, factor)

class BoostedUnit:
    def __init__(self, unit, factor): self.unit = unit; self.factor = factor
    def __call__(self):
        self.unit.energy -= 10
        return f"{self.unit.name} BOOSTED x{self.factor} = {self.unit.damage*self.factor} dmg"
    def __len__(self):  return self.unit.energy
    def __bool__(self): return self.unit.energy > 0

class BoostedSystem:
    def __init__(self, units, factor): self.units = units; self.factor = factor
    def __call__(self): return "\n".join(f"{u.name} SYSTEM x{self.factor}!" for u in self.units)
    def __len__(self):  return sum(len(u) for u in self.units)
    def __bool__(self): return any(bool(u) for u in self.units)

a2     = YoRHaUnit("A2", energy=100, damage=10)
b2     = YoRHaUnit("B2", energy=80,  damage=12)
system = a2 + b2
print((system * 2)())
print("a2 energy:", len(a2))
print("a2 active:", bool(a2))
print(a2)

## ✏️ Magic methods – Elden Ring Combat

In [ ]:
class Weapon:
    def __init__(self, name, damage): self.name = name; self.damage = damage

class Enemy:
    def __init__(self, name, hp): self.name = name; self.hp = hp

class Tarnished:
    def __init__(self, name, hp):
        self.name = name; self.hp = hp; self.weapon = None

    def __call__(self, enemy):
        dmg = self.weapon.damage if self.weapon else 5
        enemy.hp -= dmg
        if enemy.hp <= 0: return f"{self.name} defeated {enemy.name}!"
        return f"{self.name} hits {enemy.name} for {dmg} dmg! HP left: {enemy.hp}"

    def __matmul__(self, weapon):
        p = Tarnished(self.name, self.hp)
        p.weapon = weapon
        return p

    def __sub__(self, dmg):
        reduced = max(0, dmg - 10)
        self.hp -= reduced
        return f"{self.name} dodges! Takes {reduced} dmg (reduced from {dmg})"

    def __len__(self):  return self.hp
    def __bool__(self): return self.hp > 0
    def __str__(self):
        w = self.weapon.name if self.weapon else "None"
        return f"{self.name} | HP: {self.hp} | Weapon: {w}"

sword  = Weapon("Moonveil", damage=40)
enemy  = Enemy("Godrick", hp=100)
player = Tarnished("Tarnished", hp=120) @ sword

print(player(enemy))
print(player - 20)
print("HP:", len(player))
print("Alive:", bool(player))
print(player)

---
## 📋 Cheatsheet - Class 07

### singledispatch
```python
from functools import singledispatch

@singledispatch
def func(value):          # default / fallback
    return "Unknown"

@func.register(int)
def _(value): return value * 2

@func.register(str)
def _(value): return value.upper()
```

### singledispatchmethod (in classes)
```python
from functools import singledispatchmethod

class MyClass:
    @singledispatchmethod
    def handle(self, v): return "default"

    @handle.register(int)
    def _(self, v): return v * 2
```

### Full dunder reference
```python
# Representation
__str__  __repr__  __format__

# Arithmetic
__add__  __sub__  __mul__  __truediv__
__floordiv__  __mod__  __pow__  __matmul__

# Comparison
__eq__  __ne__  __lt__  __le__  __gt__  __ge__

# Container
__len__  __getitem__  __setitem__  __contains__

# Iteration
__iter__  __next__

# Callable / Context
__call__  __enter__  __exit__

# Type conversion
__int__  __float__  __bool__  __str__

# Advanced
__hash__  __copy__  __deepcopy__
```